# Opdracht 1: het huidige behandelproces in Python
Klik op een codeblok en druk op **Shift + Enter**. Werk van boven naar beneden en selecteer dezelfde Python-kernel als in het startnotebook voor data-analyse.

Dit notebook is een zelfstandige leerversie van `procesmodel.py`: je hoeft het Python-bestand niet eerst uit te voeren. Er zijn geen extra Python-pakketten nodig voor de modelcode.

**Doel:** de processtappen uit de case herkenbaar weergeven, met dezelfde codes als in het visuele model van je teamgenoot. De invoer bestaat uit zelfgekozen scenario's. De code stelt geen diagnose en voorspelt niets.

Bron: pagina 1 en opdracht 1 op pagina 2 van [de case](../../informatie/Heart%20Failure%20Prediction%20Case%20incl%20ethics.pdf). [Uitgebreide toelichting en aannames](../../informatie/opdracht1/README.md).

## Feiten en modelkeuzes
De case noemt binnenkomst via huisarts of SEH, eerste zorg door verpleegkundigen, directe IC bij ernstige gevallen met evident hartinfarct, onderzoek door specialisten en diagnose door een artsenteam. Daarna worden IC-verblijf, reguliere opname, direct vertrek, operatie en overlijden genoemd.

De case geeft geen medische criteria voor de vervolgroutes. We voeren die keuzes daarom zelf in. De volgorde van operatie en overlijden is onbekend: die tonen we als losse gebeurtenissen. De hoofdroute tot diagnose volgt de tekst; eventuele overplaatsingen en heronderzoeken zijn niet uitgewerkt. We kiezen per voorbeeld één vervolgroute; dit is een vereenvoudiging, geen ziekenhuisprotocol.

## 1. Geef elk procesonderdeel een code
Een **dictionary** koppelt een sleutel, zoals `'S01'`, aan een omschrijving. Zo gebruiken de code en het visuele model dezelfde labels. `S` duidt een activiteit of verblijfsstap aan, `D` een beslispunt en `E` een gebeurtenis.

**E01 is de startgebeurtenis:** de patiënt komt binnen. De parameter `instroom` beschrijft via welke route dat gebeurt: huisarts of SEH. De eerste zorg door verpleegkundigen is vervolgens activiteit S01. E02 is overlijden, waarvan het tijdstip niet is beschreven.

De naam `STAPPEN` blijft behouden als verzamelnaam voor alle procesonderdelen, dus ook gebeurtenissen en beslispunten. Deze codes zijn onze modelkeuze; dit is geen volledige formele BPMN-uitwerking.

Voer de cel uit. Er verschijnt nog geen uitvoer: Python bewaart de dictionary onder de naam `STAPPEN`.

In [ ]:
STAPPEN = {
    'E01': 'Binnenkomst via huisarts',
    'E02': 'Binnenkomst via spoedeisende hulp',
    'S01': 'Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie',
    'D01': 'Ernstig geval met evident hartinfarct?',
    'S02': 'Direct naar intensive care',
    'S03': 'Specialisten meten patiëntwaarden (meestal enkele uren)',
    'S04': 'Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies',
    'D02': 'Vervolgroute volgens gekozen scenario (criteria niet beschreven)',
    'S05': 'Verblijf op intensive care',
    'S06': 'Reguliere ziekenhuisopname',
    'S07': 'Ziekenhuis direct verlaten',
    'S08': 'Operatie, bijvoorbeeld plaatsing van stents',
    'E03': 'Overlijden (tijdstip in het proces onbekend)',
}

## 2. Vraag één omschrijving op
Met vierkante haken zoek je de waarde bij een sleutel op. `print()` toont die waarde. Verwacht hieronder de omschrijving van de eerste zorg door verpleegkundigen (S01).

In [2]:
print(STAPPEN['S01'])  # Output: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie

Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie


## 3. Bouw een route met een functie
Met `def` definieer je een functie: herbruikbare code die pas werkt wanneer je haar aanroept.

De invoerparameters zijn:
- `instroom`: `'huisarts'` of `'seh'`; de aankomstroute bij startgebeurtenis E01.
- `ernstig`: `True` of `False`, voor een ernstig geval met evident hartinfarct. Dit is een ingevoerde beoordeling.
- `vervolg`: `'ic'`, `'regulier'`, `'vertrek'` of `'onbekend_bij_overlijden'`.
- `operatie=False`: standaard geen operatie in het scenario; met `True` registreer je die wel.

Lees de functie in drie delen:
1. De eerste `if`-regels controleren of de invoer toegestane waarden heeft. `raise ValueError` meldt ongeldige invoer. Dit controleert geen medische geschiktheid.
2. `route` begint met startgebeurtenis E01, activiteit S01 en beslispunt D01. `append()` voegt één stap toe; `extend()` voegt meerdere stappen toe. Alleen bij `ernstig=True` wordt S02 toegevoegd.
3. `gebeurtenissen` bewaart operatie en overlijden apart. `return` geeft beide lijsten terug.

`None` betekent hier dat er geen bekende verblijf-/vertrekstap wordt toegevoegd. Bij overlijden kennen we die route niet uit de case. Ook kennen we het moment van overlijden niet; de voorbeeldroute is geen bewering dat iedereen eerst de diagnose bereikt.

In [ ]:
def modelleer_proces(instroom, ernstig, vervolg, operatie=False):
    """Geef een route en losse gebeurtenissen terug uit handmatige invoer.

    `ernstig` betekent hier exact: ernstig geval met evident hartinfarct.
    `vervolg` is ic, regulier, vertrek of onbekend_bij_overlijden.
    Een onbekende vervolgroute voorkomt dat we bij overlijden verblijf verzinnen.
    """
    if instroom not in ('huisarts', 'seh'):
        raise ValueError('Instroom moet huisarts of seh zijn.')
    if type(ernstig) is not bool or type(operatie) is not bool:
        raise ValueError('Ernstig en operatie moeten True of False zijn.')
    vervolgstappen = {
        'ic': 'S05', 'regulier': 'S06', 'vertrek': 'S07',
        'onbekend_bij_overlijden': None,
    }
    if vervolg not in vervolgstappen:
        raise ValueError('Onbekende vervolgroute.')

    if instroom == 'huisarts':
        route = ['E01', 'S01', 'D01']
    if instroom == 'seh':
        route = ['E02', 'S01', 'D01']
    if ernstig:
        route.append('S02')
    route.extend(['S03', 'S04', 'D02'])
    if vervolgstappen[vervolg] is not None:
        route.append(vervolgstappen[vervolg])

    # Deze gebeurtenissen krijgen geen verzonnen plek in de tijdlijn.
    gebeurtenissen = []
    if operatie:
        gebeurtenissen.append('S08')
    if vervolg == 'onbekend_bij_overlijden':
        gebeurtenissen.append('E03')
    return route, gebeurtenissen

## 4. Bekijk wat de functie teruggeeft
We roepen de functie aan voor een zelfgekozen voorbeeld. De twee resultaten bewaren we in `route` en `gebeurtenissen`. Verwacht een lijst zonder S02 en een lege gebeurtenissenlijst `[]`.

In [4]:
route, gebeurtenissen = modelleer_proces('huisarts', False, 'vertrek')
print('Route:', route)
print('Losse gebeurtenissen:', gebeurtenissen)

Route: ['E01', 'S01', 'D01', 'S03', 'S04', 'D02', 'S07']
Losse gebeurtenissen: []


## 5. Maak de uitvoer leesbaar
De volgende functie roept `modelleer_proces()` aan en toont daarna de omschrijvingen.

`for stap in route:` loopt één voor één door de stapcodes. `STAPPEN[stap]` haalt de omschrijving op. Een **f-string** zoals `f'{stap}'` vult de waarde van een variabele in de tekst in.

Losse gebeurtenissen krijgen nadrukkelijk het label **tijdstip onbekend**. Dat ze onder de route worden afgedrukt, betekent niet dat ze daarna plaatsvinden.

In [5]:
def toon_scenario(naam, instroom, ernstig, vervolg, operatie=False):
    route, gebeurtenissen = modelleer_proces(instroom, ernstig, vervolg, operatie)
    print(f'\n{naam} | instroom: {instroom} | ernstig: {ernstig}')
    for stap in route:
        print(f'  {stap}: {STAPPEN[stap]}')
    for stap in gebeurtenissen:
        print(f'  Losse gebeurtenis, tijdstip onbekend - {stap}: {STAPPEN[stap]}')

## 6. Voorbeeld A: via de huisarts, daarna vertrek
Dit is een gekozen procesroute, geen advies om een patiënt te ontslaan. Omdat `ernstig=False`, verschijnt S02 niet. Het gekozen vervolg `'vertrek'` geeft S07.

In [6]:
toon_scenario('Voorbeeld A', 'huisarts', False, 'vertrek')


Voorbeeld A | instroom: huisarts | ernstig: False
  E01: Binnenkomst via huisarts of spoedeisende hulp
  S01: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie
  D01: Ernstig geval met evident hartinfarct?
  S03: Specialisten meten patiëntwaarden (meestal enkele uren)
  S04: Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies
  D02: Vervolgroute volgens gekozen scenario (criteria niet beschreven)
  S07: Ziekenhuis direct verlaten


## 7. Voorbeeld B: directe IC en een operatie
Bij `ernstig=True` verschijnt S02 vóór het onderzoek. Het vervolg `'ic'` voegt S05 toe: IC-verblijf na de diagnose in dit scenario. S02 en S05 hebben dus een verschillende betekenis.

`operatie=True` registreert S08 apart. De case specificeert de timing van die operatie niet.

In [7]:
toon_scenario('Voorbeeld B', 'seh', True, 'ic', operatie=True)


Voorbeeld B | instroom: seh | ernstig: True
  E01: Binnenkomst via huisarts of spoedeisende hulp
  S01: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie
  D01: Ernstig geval met evident hartinfarct?
  S02: Direct naar intensive care
  S03: Specialisten meten patiëntwaarden (meestal enkele uren)
  S04: Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies
  D02: Vervolgroute volgens gekozen scenario (criteria niet beschreven)
  S05: Verblijf op intensive care
  Losse gebeurtenis, tijdstip onbekend - S08: Operatie, bijvoorbeeld plaatsing van stents


## 8. Voorbeeld C: reguliere opname
Verwacht S06 als vervolg. Vergelijk deze uitvoer met voorbeeld A: dezelfde beginroute, maar een andere ingevoerde vervolgkeuze.

In [8]:
toon_scenario('Voorbeeld C', 'huisarts', False, 'regulier')


Voorbeeld C | instroom: huisarts | ernstig: False
  E01: Binnenkomst via huisarts of spoedeisende hulp
  S01: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie
  D01: Ernstig geval met evident hartinfarct?
  S03: Specialisten meten patiëntwaarden (meestal enkele uren)
  S04: Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies
  D02: Vervolgroute volgens gekozen scenario (criteria niet beschreven)
  S06: Reguliere ziekenhuisopname


## 9. Voorbeeld D: overlijden genoemd, route onvolledig bekend
`onbekend_bij_overlijden` is onze technische markering. Overlijden is geen behandelbeslissing. We kennen de plaats in het proces niet en voegen geen verzonnen verblijfroute toe.

Dit eenvoudige voorbeeld toont wel de gebruikelijke hoofdroute; het legt niet vast welke stappen werkelijk aan een overlijden voorafgaan. Het model is op dit punt bewust onvolledig.

In [9]:
toon_scenario('Voorbeeld D', 'seh', True, 'onbekend_bij_overlijden')


Voorbeeld D | instroom: seh | ernstig: True
  E01: Binnenkomst via huisarts of spoedeisende hulp
  S01: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie
  D01: Ernstig geval met evident hartinfarct?
  S02: Direct naar intensive care
  S03: Specialisten meten patiëntwaarden (meestal enkele uren)
  S04: Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies
  D02: Vervolgroute volgens gekozen scenario (criteria niet beschreven)
  Losse gebeurtenis, tijdstip onbekend - E02: Overlijden (tijdstip in het proces onbekend)


## 10. Probeer zelf een wijziging
Verander hieronder alleen `'regulier'` in `'ic'`. Voorspel eerst de uitvoer en voer daarna de cel uit.

Welke stap verandert? Verschijnt S02 ook? Leg uit welke invoer S02 bepaalt en welke invoer S05 bepaalt.

In [10]:
toon_scenario('Mijn oefening', 'huisarts', False, 'ic')


Mijn oefening | instroom: huisarts | ernstig: False
  E01: Binnenkomst via huisarts of spoedeisende hulp
  S01: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie
  D01: Ernstig geval met evident hartinfarct?
  S03: Specialisten meten patiëntwaarden (meestal enkele uren)
  S04: Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies
  D02: Vervolgroute volgens gekozen scenario (criteria niet beschreven)
  S05: Verblijf op intensive care


## Mijn bevindingen
Dubbelklik op deze tekst, vul je antwoorden in en druk op **Shift + Enter**.

- Ik verwachtte dat S02 niet verschijnt. 
- In de uitvoer veranderde ...
- S02 wordt bepaald door ...
- S05 wordt bepaald door ...
- Een aanname die ik met mijn teamgenoot wil bespreken is ...

## Wat je nu kunt toelichten
Je gebruikt een dictionary voor stapomschrijvingen, lijsten voor routes en een functie met `if`-regels voor vooraf gekozen vertakkingen. De code berekent geen medische beslissingen. Dezelfde stapcodes kunnen in het visuele model staan.

Dit notebook en `procesmodel.py` zijn afzonderlijk uitvoerbare versies. Wijzigingen worden niet automatisch tussen beide gesynchroniseerd. Gebruik het notebook om te leren en stem inhoudelijke wijzigingen later af met het script.